In [1]:
import os
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch import tensor
import numpy as np
import csv
from scipy.signal import savgol_filter


input_size    = 45
hidden_layers = [256, 128, 64]
output_size   = 35
dropout_probs = [0.0, 0.0, 0.0]
lr            = 1e-4
batch_size    = 256

print(input_size, hidden_layers, output_size, dropout_probs)


directory = 'Figures/['
for a in range(len(hidden_layers)):
    directory += str(hidden_layers[a]) + ','
    if a == len(hidden_layers) - 1:
        directory = directory[:-1]
        directory += ']'
if not os.path.isdir(directory):
    os.makedirs(directory)
out = directory


class CustomDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir      = data_dir
        all_intensities    = []
        all_first_derivs   = []
        all_second_derivs  = []
        all_smoothed       = []
        all_labels         = []

        for filename in os.listdir(self.data_dir):
            if filename.endswith('_energy.pkl'):
                label_str = filename.split('_')[0][1:]
                label     = int(label_str)

                energy_path    = os.path.join(self.data_dir, filename)
                intensity_path = os.path.join(self.data_dir, f'P{label_str}_intensity.pkl')

                if os.path.exists(intensity_path):
                    with open(energy_path, 'rb') as f:
                        e_data = np.array(pickle.load(f))
                    with open(intensity_path, 'rb') as f:
                        i_data = np.array(pickle.load(f))

                    if e_data.shape == i_data.shape:
                        all_intensities.append(i_data)
                        all_labels.append(np.full(i_data.shape[0], label))

                        # 1st derivative: dI/dE using actual energy spacing
                        d1_data = np.array([
                            np.gradient(i_data[k], e_data[k])
                            for k in range(i_data.shape[0])
                        ])
                        all_first_derivs.append(d1_data)

                        # 2nd derivative: d²I/dE²
                        d2_data = np.array([
                            np.gradient(d1_data[k], e_data[k])
                            for k in range(d1_data.shape[0])
                        ])
                        all_second_derivs.append(d2_data)

                        # Savitzky-Golay smoothed — window=7, polyorder=3
                        sg_data = np.array([
                            savgol_filter(i_data[k], window_length=7, polyorder=3)
                            for k in range(i_data.shape[0])
                        ])
                        all_smoothed.append(sg_data)

        full_intensity     = np.concatenate(all_intensities,   axis=0)
        full_first_derivs  = np.concatenate(all_first_derivs,  axis=0)
        full_second_derivs = np.concatenate(all_second_derivs, axis=0)
        full_smoothed      = np.concatenate(all_smoothed,      axis=0)
        full_labels        = np.concatenate(all_labels,        axis=0)

        def zscore(arr):
            mean = arr.mean(axis=1, keepdims=True)
            std  = arr.std(axis=1,  keepdims=True) + 1e-8
            return (arr - mean) / std

        self.intensity_tensors    = torch.from_numpy(zscore(full_intensity)).float()
        self.first_deriv_tensors  = torch.from_numpy(zscore(full_first_derivs)).float()
        self.second_deriv_tensors = torch.from_numpy(zscore(full_second_derivs)).float()
        self.smoothed_tensors     = torch.from_numpy(zscore(full_smoothed)).float()
        self.labels               = torch.from_numpy(full_labels).long()

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        # Ch 0: raw intensity   — anchor lineshape
        # Ch 1: dI/dE           — peak positions and slopes
        # Ch 2: d²I/dE²         — curvature and shoulders
        # Ch 3: SG smoothed     — denoised bridge between training and inference
        input_tensor = torch.stack((
            self.intensity_tensors[idx],
            self.first_deriv_tensors[idx],
            self.second_deriv_tensors[idx],
            self.smoothed_tensors[idx],
        ), dim=0)
        return input_tensor, self.labels[idx]


45 [256, 128, 64] 35 [0.0, 0.0, 0.0]


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleDNN(nn.Module):
    def __init__(self, output_size=output_size, hidden_layers=hidden_layers):
        super(SimpleDNN, self).__init__()

        # Convolutional Layers — 4 input channels
        self.conv1 = nn.Conv1d(4, 8,  kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(8, 16, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(16, 32, kernel_size=5, padding=2)

        # flatten_size: 32 channels * input_size (no pooling)
        self.flatten_size = 32 * input_size

        self.fc1 = nn.Linear(self.flatten_size, hidden_layers[0])
        self.fc2 = nn.Linear(hidden_layers[0],  hidden_layers[1])
        self.fc3 = nn.Linear(hidden_layers[1],  hidden_layers[2])
        self.fc4 = nn.Linear(hidden_layers[2],  output_size)

        self.dropout = nn.Dropout(p=0.0)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.view(x.size(0), -1)  # (batch, 32 * input_size)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)

        return self.fc4(x)


In [3]:
import time
import os
import csv
import torch

def train_gpu_resident(model, train_loader, val_loader, criterion, optimizer, scheduler,
                       num_epochs, batch_size,
                       checkpoint_dir=directory + '/checkpoints',
                       metrics_file=directory + '/training_metrics.csv'):

    best_val_loss     = float('inf')
    epochs_no_improve = 0
    min_delta         = 0.0020
    device            = next(model.parameters()).device

    os.makedirs(checkpoint_dir, exist_ok=True)

    with open(metrics_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train Loss', 'Train Accuracy', 'Val Loss', 'Val Accuracy', 'LR'])

        for epoch in range(num_epochs):
            start_time = time.time()

            # --- Training Phase ---
            model.train()
            running_loss, correct_train, total_train, num_batches = 0.0, 0, 0, 0

            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss    = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss  += loss.item()
                _, predicted   = torch.max(outputs.data, 1)
                total_train   += labels.size(0)
                correct_train += (predicted == labels).sum().item()
                num_batches   += 1

            train_loss     = running_loss / num_batches
            train_accuracy = 100 * correct_train / total_train
            vram_used      = torch.cuda.memory_reserved() / 1e6 if torch.cuda.is_available() else 0

            # --- Validation Phase ---
            model.eval()
            val_loss_total, correct_val, total_val = 0.0, 0, 0

            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels  = inputs.to(device), labels.to(device)
                    val_outputs     = model(inputs)
                    val_loss_total += criterion(val_outputs, labels).item()
                    _, predicted    = torch.max(val_outputs.data, 1)
                    total_val      += labels.size(0)
                    correct_val    += (predicted == labels).sum().item()

            val_loss     = val_loss_total / len(val_loader)
            val_accuracy = 100 * correct_val / total_val

            # --- Scheduler & LR Tracking ---
            # scheduler.step(val_loss)
            current_lr     = optimizer.param_groups[0]['lr']
            epoch_duration = time.time() - start_time

            writer.writerow([epoch + 1, train_loss, train_accuracy, val_loss, val_accuracy, current_lr])
            f.flush()

            print(f'Epoch [{epoch+1}/{num_epochs}] | Time: {epoch_duration:.2f}s | VRAM: {vram_used:.0f}MB | '
                  f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
                  f'Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}% | '
                  f'LR: {current_lr:.6f} | Patience: {epochs_no_improve}')

            # --- Best Model & Early Stopping ---
            if val_loss < best_val_loss - min_delta:
                best_val_loss     = val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best_model.pth'))
                print(f'  ^ Best model saved (val_loss: {val_loss:.4f})')
            else:
                epochs_no_improve += 1


In [4]:
if __name__ == '__main__':
    data_dir = 'Datasets'
    dataset  = CustomDataset(data_dir)

    if len(dataset) == 0:
        print('No data found.')
    else:
        train_size = int(0.8 * len(dataset))
        val_size   = len(dataset) - train_size
        train_ds, val_ds = random_split(
            dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )

        os.makedirs(directory, exist_ok=True)

        # Save val_loader for inference
        import pickle
        val_loader_save = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        with open(f'{directory}/val_loader.pkl', 'wb') as f:
            pickle.dump(val_loader_save, f)
        print("Validation DataLoader saved to 'val_loader.pkl'.")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {device}')

        # DataLoader-based — avoids OOM from loading full dataset to VRAM
        train_loader = DataLoader(
            train_ds, batch_size=batch_size,
            shuffle=True, num_workers=0, pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size,
            shuffle=False, num_workers=0, pin_memory=True
        )

        model     = SimpleDNN().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=20,
            min_lr=1e-5,
            threshold=0.0020,
            threshold_mode='abs'
        )

        train_gpu_resident(
            model, train_loader, val_loader,
            criterion, optimizer, scheduler,
            num_epochs=1000, batch_size=batch_size
        )


Validation DataLoader saved to 'val_loader.pkl'.
Moving dataset to cuda...
Epoch [1/1000]| Time: 10.97s | VRAM: 178MB | Train Loss: 3.2511 | Val Loss: 3.0472 | Train Acc: 10.02% | Val Acc: 12.48% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [2/1000]| Time: 10.43s | VRAM: 593MB | Train Loss: 2.8736 | Val Loss: 2.7454 | Train Acc: 16.43% | Val Acc: 18.98% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [3/1000]| Time: 10.32s | VRAM: 593MB | Train Loss: 2.6021 | Val Loss: 2.4790 | Train Acc: 22.33% | Val Acc: 24.91% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [4/1000]| Time: 9.92s | VRAM: 593MB | Train Loss: 2.3687 | Val Loss: 2.2803 | Train Acc: 27.52% | Val Acc: 30.03% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [5/1000]| Time: 10.26s | VRAM: 593MB | Train Loss: 2.2246 | Val Loss: 2.1804 | Train Acc: 31.22% | Val Acc: 32.32% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [6/1000]| Time: 9.63s | VRAM: 593MB | Train Loss: 2.1321 | Val Loss: 2.0765 | Train A

KeyboardInterrupt: 

In [ ]:

content = (
    f"input_size = {input_size}\n"
    f"hidden_layers = {hidden_layers}\n"
    f"output_size = {output_size}\n"
    f"dropout_probs = {dropout_probs}\n"
    f"lr = {lr}\n"
)

# Write to a file
with open(directory+"/model_params.txt", "w") as f:
    f.write(content)

print("File written successfully!")